# Project 2: House Price Prediction System
**MCA III Semester – Lab Sheet-06: Mini Projects on Supervised Learning Applications**

**Problem statement:** Predict the price of a house (in lakhs) from area, number of bedrooms, bathrooms, parking spaces, property age and location.

**Type of problem:** Supervised learning – **Regression**

**Workflow:** Load data → Preprocess → EDA → Train/Test split → Compare regression algorithms → Evaluate → Visualize → Save model → Predict

In [ ]:
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 1. Load the Dataset

In [ ]:
house_df = pd.read_csv("house_price_dataset.csv")
print("Shape:", house_df.shape)
house_df.head()

In [ ]:
house_df.info()

In [ ]:
house_df.describe().T

## 2. Data Preprocessing
### 2.1 Missing values and duplicates
Numeric columns will be filled with the **median**, the categorical column (`location`) with the **most frequent** value. Imputation is done inside a pipeline after the train/test split.

In [ ]:
print("Missing values per column:")
print(house_df.isnull().sum())
print("\nDuplicate rows:", house_df.duplicated().sum())
house_df = house_df.drop_duplicates()

### 2.2 Outlier check

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(y=house_df["area_sqft"], ax=axes[0], color="skyblue")
axes[0].set_title("Area (sq ft)")
sns.boxplot(y=house_df["price_lakhs"], ax=axes[1], color="salmon")
axes[1].set_title("Price (lakhs)")
plt.tight_layout()
plt.show()

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(house_df["price_lakhs"], bins=30, kde=True, ax=axes[0], color="purple")
axes[0].set_title("Distribution of House Price")
sns.scatterplot(data=house_df, x="area_sqft", y="price_lakhs", hue="location", alpha=0.6, ax=axes[1])
axes[1].set_title("Area vs Price by Location")
plt.tight_layout()
plt.show()

In [ ]:
location_order = house_df.groupby("location")["price_lakhs"].median().sort_values(ascending=False).index
plt.figure(figsize=(8, 4))
sns.boxplot(data=house_df, x="location", y="price_lakhs", order=location_order)
plt.title("Price by Location")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(house_df.select_dtypes("number").corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

**Observation:** Area is the strongest numeric driver of price; central locations are the most expensive; older properties tend to be cheaper.

## 4. Train / Test Split

In [ ]:
X = house_df.drop(columns="price_lakhs")
y = house_df["price_lakhs"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])

## 5. Preprocessing Pipeline
- **Numeric features:** median imputation → standard scaling
- **Categorical feature (`location`):** most-frequent imputation → one-hot encoding

In [ ]:
numeric_features = ["area_sqft", "bedrooms", "bathrooms", "parking_spaces", "property_age_years"]
categorical_features = ["location"]

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_features),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_features),
])


def build_pipeline(model):
    return Pipeline([("preprocessor", preprocessor), ("model", model)])

## 6. Compare Regression Algorithms

In [ ]:
candidate_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Decision Tree": DecisionTreeRegressor(max_depth=8, random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}


def evaluate_model(name, pipeline):
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    cv_r2 = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="r2").mean()
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_test, predictions),
        "CV R2 (5-fold)": cv_r2,
    }


fitted_pipelines = {}
results = []
for model_name, model in candidate_models.items():
    pipeline = build_pipeline(model)
    results.append(evaluate_model(model_name, pipeline))
    fitted_pipelines[model_name] = pipeline

results_df = pd.DataFrame(results).sort_values("R2", ascending=False).reset_index(drop=True)
results_df.round(4)

### 6.1 Hyperparameter tuning (Gradient Boosting)

In [ ]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [2, 3],
}
grid_search = GridSearchCV(build_pipeline(GradientBoostingRegressor(random_state=RANDOM_STATE)),
                           param_grid, cv=5, scoring="r2", n_jobs=-1)
grid_search.fit(X_train, y_train)
print("Best parameters:", grid_search.best_params_)

tuned_pipeline = grid_search.best_estimator_
fitted_pipelines["Gradient Boosting (Tuned)"] = tuned_pipeline
tuned_predictions = tuned_pipeline.predict(X_test)
tuned_mse = mean_squared_error(y_test, tuned_predictions)
results_df.loc[len(results_df)] = {
    "Model": "Gradient Boosting (Tuned)",
    "MAE": mean_absolute_error(y_test, tuned_predictions),
    "MSE": tuned_mse,
    "RMSE": np.sqrt(tuned_mse),
    "R2": r2_score(y_test, tuned_predictions),
    "CV R2 (5-fold)": grid_search.best_score_,
}
results_df = results_df.sort_values("R2", ascending=False).reset_index(drop=True)
results_df.round(4)

## 7. Visualization of Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.barplot(data=results_df, x="R2", y="Model", ax=axes[0], palette="viridis")
axes[0].set_title("R² Score (higher is better)")
sns.barplot(data=results_df, x="RMSE", y="Model", ax=axes[1], palette="magma")
axes[1].set_title("RMSE in lakhs (lower is better)")
plt.tight_layout()
plt.show()

In [ ]:
best_model_name = results_df.loc[0, "Model"]
best_pipeline = fitted_pipelines[best_model_name]
best_predictions = best_pipeline.predict(X_test)
residuals = y_test - best_predictions
print("Best model:", best_model_name)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test, best_predictions, alpha=0.6, color="royalblue")
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], "r--", label="Perfect prediction")
axes[0].set_xlabel("Actual Price (lakhs)")
axes[0].set_ylabel("Predicted Price (lakhs)")
axes[0].set_title("Actual vs Predicted")
axes[0].legend()

sns.histplot(residuals, bins=30, kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("Residual Distribution")
axes[1].set_xlabel("Residual (Actual - Predicted)")
plt.tight_layout()
plt.show()

In [ ]:
final_estimator = best_pipeline.named_steps["model"]
feature_names = best_pipeline.named_steps["preprocessor"].get_feature_names_out()
feature_names = [name.split("__")[-1] for name in feature_names]

if hasattr(final_estimator, "feature_importances_"):
    importance_values = final_estimator.feature_importances_
    importance_title = "Feature Importance"
else:
    importance_values = np.abs(final_estimator.coef_)
    importance_title = "Absolute Coefficient (scaled features)"

pd.Series(importance_values, index=feature_names).sort_values().plot(
    kind="barh", figsize=(8, 5), color="seagreen")
plt.title(importance_title)
plt.show()

## 8. Interpretation of Evaluation Metrics
- **MAE** – average absolute error in price (lakhs).
- **MSE / RMSE** – penalise large errors more; RMSE is in lakhs.
- **R² score** – share of price variation explained by the model.
- **CV R²** – average R² across 5 training folds; a small gap from test R² indicates good generalisation.
- **Model comparison:** Linear models capture the roughly linear price trends, whereas tree-based ensembles can also learn non-linear effects and feature interactions (e.g. area × location). A single decision tree is easy to interpret but tends to overfit.

In [ ]:
best_row = results_df.iloc[0]
print(f"Best model : {best_row['Model']}")
print(f"R2 score   : {best_row['R2']:.3f} -> explains {best_row['R2'] * 100:.1f}% of the variation in prices.")
print(f"MAE        : {best_row['MAE']:.2f} lakhs on average")
print(f"RMSE       : {best_row['RMSE']:.2f} lakhs")
gap = abs(best_row["R2"] - best_row["CV R2 (5-fold)"])
print(f"Test R2 vs CV R2 gap: {gap:.3f} -> {'good generalisation' if gap < 0.10 else 'possible over/under-fitting'}")

## 9. Save the Trained Model
The full pipeline (preprocessing + model) is saved with Joblib.

In [ ]:
MODEL_FILENAME = "house_price_model.joblib"
joblib.dump(best_pipeline, MODEL_FILENAME)
print("Model saved as", MODEL_FILENAME)

## 10. Load the Model and Predict for a New House

In [ ]:
loaded_model = joblib.load(MODEL_FILENAME)

new_house = pd.DataFrame([{
    "area_sqft": 1800,
    "bedrooms": 3,
    "bathrooms": 2,
    "parking_spaces": 1,
    "property_age_years": 5,
    "location": "Riverside",
}])
predicted_price = loaded_model.predict(new_house)[0]
print(f"Predicted price: {predicted_price:.2f} lakhs")

## 11. Conclusion
- A complete regression workflow was built with preprocessing pipelines, model comparison, tuning, evaluation and model persistence.
- Ensemble / tuned models are compared against linear baselines using MAE, RMSE and R².
- Area and location are the most influential features for house price.